# History, Audit And Temporal Data

## Quick notes
### What this covers
- Keeping the past instead of overwriting it: soft delete, audit-log triggers, history tables, and "valid from / valid to" rows (SCD type 2).
- Asking questions about the past: "what was true on this date" (as-of queries), who changed what, current-row views.
- Protecting history: overlap guards, optimistic locking with a version column, an append-only ledger with a running balance.
- Erasing data on purpose (GDPR style): hard erase vs anonymise. A short note on bitemporal data.
### Key points
- Soft delete = a `deleted_at` column instead of a real `DELETE`. Every read must then filter on it (a view helps).
- An audit log records WHO did WHAT and WHEN, usually filled by triggers so no code path can skip it.
- A history table keeps full old copies of a row; an audit log keeps events. Pick by the question you need to answer.
- SCD type 2: each version of a thing is a row with `valid_from` and `valid_to`. Use half-open ranges: `valid_from <= d AND d < valid_to`.
- "As of date d" = the row whose range contains d. One open-ended sentinel (`9999-12-31`) marks the current row.
- Optimistic locking: `UPDATE ... WHERE id = ? AND version = ?`, then check `changes()`. Zero rows changed means someone else got there first.
- Money and stock belong in an append-only ledger: never edit, add a reversing entry. Balance = running sum.
### Gotchas (the part worth revising twice)
- Forgetting `deleted_at IS NULL` in one query leaks deleted rows. Ordinary `UNIQUE` also blocks re-creating a soft-deleted value: use a partial unique index.
- Closed-range intervals (`BETWEEN`) double-count the shared boundary day. Use half-open ranges.
- Two rows for the same key with overlapping ranges make "as of" return duplicates. Guard with a trigger.
- Triggers cannot know the current user or date by themselves in a portable way; this notebook reads both from a tiny `ctx` table so results are repeatable.
- Audit logs and history tables copy personal data. An "erase" request must reach them too (and backups).
- Undeleting can collide with rows created after the delete (unique keys).
### When it matters
Anything with accountability or time: prices, addresses, salaries, permissions, accounting, customer records, regulated data.
### One-line recap example
```sql
SELECT city FROM customer_addr WHERE customer_id = 1 AND valid_from <= '2025-06-01' AND '2025-06-01' < valid_to;
```

## 01. Seed data and the ctx table
- `ctx` holds the "current actor" and the "current date". Triggers read it, so every demo is repeatable (no clock calls). In a real app you would set these per request.
- `users` has a `deleted_at` column for soft delete, plus a partial unique index: email must be unique only among live rows.
- Dates are fixed text in `YYYY-MM-DD` form, which sorts correctly as text.

In [1]:
# setup: an in-memory SQLite database and a %%sql cell magic that prints result tables
import sqlite3
from IPython.core.magic import register_cell_magic

con = sqlite3.connect(":memory:", isolation_level=None)   # autocommit, so BEGIN/COMMIT work as written
con.execute("PRAGMA foreign_keys = ON")

def _table(cols, rows):
    cells = [[("NULL" if v is None else str(v)) for v in r] for r in rows]
    widths = [max([len(c)] + [len(r[i]) for r in cells]) for i, c in enumerate(cols)]
    line = lambda vals: " | ".join(v.ljust(w) for v, w in zip(vals, widths))
    print(line(cols)); print("-+-".join("-" * w for w in widths))
    for r in cells: print(line(r))
    print(f"({len(rows)} row{'s' if len(rows) != 1 else ''})\n")

def _statements(text):
    """Split a cell into statements (handles several per line, triggers, quoted semicolons)."""
    out, start = [], 0
    for i, ch in enumerate(text):
        if ch == ";" and sqlite3.complete_statement(text[start:i + 1]):
            out.append(text[start:i + 1].strip()); start = i + 1
    if text[start:].strip():
        out.append(text[start:].strip())
    return out

@register_cell_magic
def sql(line, cell):
    """Run every statement in the cell; print rows for statements that return them."""
    expect_error = "expect_error" in line
    for stmt in _statements(cell):
        try:
            cur = con.execute(stmt)
            if cur.description:
                _table([d[0] for d in cur.description], cur.fetchall())
        except sqlite3.Error as exc:
            if not expect_error:
                raise
            print(f"ERROR: {exc}\n")

In [2]:
%%sql
CREATE TABLE ctx (k TEXT PRIMARY KEY, v TEXT NOT NULL);
INSERT INTO ctx VALUES ('actor', 'asha'), ('now', '2026-01-05');
CREATE TABLE users (
    user_id    INTEGER PRIMARY KEY,
    email      TEXT NOT NULL,
    name       TEXT NOT NULL,
    deleted_at TEXT
);
CREATE UNIQUE INDEX ux_users_email_live ON users(email) WHERE deleted_at IS NULL;
INSERT INTO users (user_id, email, name) VALUES
    (1, 'asha@example.com', 'Asha Rao'), (2, 'ben@example.com', 'Ben Carter'), (3, 'chen@example.com', 'Chen Wei');
SELECT user_id, email, name, deleted_at FROM users ORDER BY user_id;

user_id | email            | name       | deleted_at
--------+------------------+------------+-----------
1       | asha@example.com | Asha Rao   | NULL      
2       | ben@example.com  | Ben Carter | NULL      
3       | chen@example.com | Chen Wei   | NULL      
(3 rows)



## 02. Soft delete
- Instead of `DELETE`, stamp `deleted_at`. The row, its id and everything pointing at it stay intact, and the delete can be undone.
- Every reader must now add `WHERE deleted_at IS NULL`. Put that in a view (`v_users_active`) and let normal code query the view only.
- Bad: the plain table query still returns Ben. Good: the view hides him.
- The partial unique index means a new live "ben@example.com" is allowed after the old one was soft-deleted, but two live ones are not.

In [3]:
%%sql
UPDATE users SET deleted_at = (SELECT v FROM ctx WHERE k = 'now') WHERE user_id = 2;
CREATE VIEW v_users_active AS SELECT user_id, email, name FROM users WHERE deleted_at IS NULL;
SELECT user_id, name FROM users ORDER BY user_id;
SELECT user_id, name FROM v_users_active ORDER BY user_id;
INSERT INTO users (user_id, email, name) VALUES (4, 'ben@example.com', 'Ben The Second');
SELECT user_id, email, name FROM v_users_active ORDER BY user_id;

user_id | name      
--------+-----------
1       | Asha Rao  
2       | Ben Carter
3       | Chen Wei  
(3 rows)

user_id | name    
--------+---------
1       | Asha Rao
3       | Chen Wei
(2 rows)

user_id | email            | name          
--------+------------------+---------------
1       | asha@example.com | Asha Rao      
3       | chen@example.com | Chen Wei      
4       | ben@example.com  | Ben The Second
(3 rows)



In [4]:
%%sql expect_error
INSERT INTO users (user_id, email, name) VALUES (5, 'asha@example.com', 'Imposter');

ERROR: UNIQUE constraint failed: users.email



## 03. Undelete (and when it collides)
- Undelete = set `deleted_at` back to NULL. One statement.
- But the world moved on: user 4 now owns `ben@example.com`. Restoring user 2 would create two live rows with the same email, and the partial index refuses.
- Resolve the conflict first (here: soft-delete the newcomer), then restore. Real systems decide this by business rule, not by SQL.

In [5]:
%%sql expect_error
UPDATE users SET deleted_at = NULL WHERE user_id = 2;

ERROR: UNIQUE constraint failed: users.email



In [6]:
%%sql
UPDATE users SET deleted_at = (SELECT v FROM ctx WHERE k = 'now') WHERE user_id = 4;
UPDATE users SET deleted_at = NULL WHERE user_id = 2;
SELECT user_id, email, name FROM v_users_active ORDER BY user_id;

user_id | email            | name      
--------+------------------+-----------
1       | asha@example.com | Asha Rao  
2       | ben@example.com  | Ben Carter
3       | chen@example.com | Chen Wei  
(3 rows)



## 04. Audit-log triggers
- One generic `audit_log` table: which table and row, what action, who, when, and the old and new row as JSON.
- Three triggers (insert, update, delete) fill it automatically. Application code cannot forget.
- The update trigger has a `WHEN` clause so no-op updates (same values written again) create no noise.
- `OLD` is the row before, `NEW` the row after. Insert has only `NEW`, delete only `OLD`. See 05-DDL-And-Integrity/02-Views-And-Triggers.
- Below we run a short story: three inserts, a price change, a no-op, a rename plus price change, and a delete.

In [7]:
%%sql
CREATE TABLE products (product_id INTEGER PRIMARY KEY, name TEXT NOT NULL, price_cents INTEGER NOT NULL);
CREATE TABLE audit_log (
    audit_id   INTEGER PRIMARY KEY AUTOINCREMENT,
    table_name TEXT NOT NULL, row_id INTEGER NOT NULL, action TEXT NOT NULL,
    actor TEXT NOT NULL, at TEXT NOT NULL, old_row TEXT, new_row TEXT
);
CREATE TRIGGER trg_products_audit_ins AFTER INSERT ON products BEGIN
    INSERT INTO audit_log (table_name, row_id, action, actor, at, old_row, new_row)
    VALUES ('products', NEW.product_id, 'INSERT', (SELECT v FROM ctx WHERE k = 'actor'), (SELECT v FROM ctx WHERE k = 'now'),
            NULL, json_object('name', NEW.name, 'price_cents', NEW.price_cents));
END;
CREATE TRIGGER trg_products_audit_upd AFTER UPDATE ON products
WHEN OLD.name IS NOT NEW.name OR OLD.price_cents IS NOT NEW.price_cents BEGIN
    INSERT INTO audit_log (table_name, row_id, action, actor, at, old_row, new_row)
    VALUES ('products', NEW.product_id, 'UPDATE', (SELECT v FROM ctx WHERE k = 'actor'), (SELECT v FROM ctx WHERE k = 'now'),
            json_object('name', OLD.name, 'price_cents', OLD.price_cents),
            json_object('name', NEW.name, 'price_cents', NEW.price_cents));
END;
CREATE TRIGGER trg_products_audit_del AFTER DELETE ON products BEGIN
    INSERT INTO audit_log (table_name, row_id, action, actor, at, old_row, new_row)
    VALUES ('products', OLD.product_id, 'DELETE', (SELECT v FROM ctx WHERE k = 'actor'), (SELECT v FROM ctx WHERE k = 'now'),
            json_object('name', OLD.name, 'price_cents', OLD.price_cents), NULL);
END;
INSERT INTO products VALUES (1, 'Widget', 1000), (2, 'Gadget', 2500), (3, 'Gizmo', 400);
UPDATE ctx SET v = '2026-01-06' WHERE k = 'now';
UPDATE ctx SET v = 'ben' WHERE k = 'actor';
UPDATE products SET price_cents = 1200 WHERE product_id = 1;
UPDATE products SET price_cents = 2500 WHERE product_id = 2;
UPDATE ctx SET v = '2026-01-07' WHERE k = 'now';
UPDATE ctx SET v = 'asha' WHERE k = 'actor';
UPDATE products SET name = 'Gadget Pro', price_cents = 2800 WHERE product_id = 2;
UPDATE ctx SET v = '2026-01-08' WHERE k = 'now';
UPDATE ctx SET v = 'ben' WHERE k = 'actor';
DELETE FROM products WHERE product_id = 3;
SELECT count(*) AS audit_rows FROM audit_log;

audit_rows
----------
6         
(1 row)



## 05. Reading the audit log
- Six audit rows: 3 inserts, 2 real updates, 1 delete. The no-op update (price rewritten with the same value) wrote nothing, thanks to the `WHEN` clause.
- History of one row: filter by `table_name` and `row_id`, order by `audit_id` (a steady counter, so it is the true order of events).
- "Who changed the price?" compare `json_extract(old_row, '$.price_cents')` with the new one. `IS NOT` handles NULLs.
- State as of a date: for each row take the last event at or before that date, and ignore rows whose last event was a delete.

In [8]:
%%sql
SELECT audit_id, at, actor, action, old_row, new_row FROM audit_log WHERE row_id = 2 ORDER BY audit_id;
SELECT row_id, at, actor,
       json_extract(old_row, '$.price_cents') AS from_price,
       json_extract(new_row, '$.price_cents') AS to_price
FROM audit_log
WHERE action = 'UPDATE' AND json_extract(old_row, '$.price_cents') IS NOT json_extract(new_row, '$.price_cents')
ORDER BY audit_id;
SELECT row_id, json_extract(new_row, '$.name') AS name, json_extract(new_row, '$.price_cents') AS price_cents
FROM (SELECT *, row_number() OVER (PARTITION BY row_id ORDER BY audit_id DESC) AS rn
      FROM audit_log WHERE table_name = 'products' AND at <= '2026-01-06')
WHERE rn = 1 AND action <> 'DELETE'
ORDER BY row_id;

audit_id | at         | actor | action | old_row                              | new_row                                 
---------+------------+-------+--------+--------------------------------------+-----------------------------------------
2        | 2026-01-05 | asha  | INSERT | NULL                                 | {"name":"Gadget","price_cents":2500}    
5        | 2026-01-07 | asha  | UPDATE | {"name":"Gadget","price_cents":2500} | {"name":"Gadget Pro","price_cents":2800}
(2 rows)

row_id | at         | actor | from_price | to_price
-------+------------+-------+------------+---------
1      | 2026-01-06 | ben   | 1000       | 1200    
2      | 2026-01-07 | asha  | 2500       | 2800    
(2 rows)

row_id | name   | price_cents
-------+--------+------------
1      | Widget | 1200       
2      | Gadget | 2500       
3      | Gizmo  | 400        
(3 rows)



## 06. The history-table pattern
- Alternative to a JSON log: a typed `employees_history` table with the same columns, plus the time range each version was valid (`valid_from`, `valid_to`) and the operation.
- Triggers copy the OLD row into history whenever a row is updated or deleted. The live table stays small and current.
- Pros over a JSON audit log: normal columns, normal indexes, easy joins and SQL on old versions. Cons: one history table per table, and schema changes must touch both (see 05-DDL-And-Integrity/03-Schema-Evolution-And-Migrations).
- Here the application stamps `updated_at` on each change; the trigger uses it as the old version's end date.

In [9]:
%%sql
CREATE TABLE employees (emp_id INTEGER PRIMARY KEY, name TEXT NOT NULL, title TEXT NOT NULL, salary INTEGER NOT NULL, updated_at TEXT NOT NULL);
CREATE TABLE employees_history (
    hist_id INTEGER PRIMARY KEY AUTOINCREMENT,
    emp_id INTEGER NOT NULL, name TEXT NOT NULL, title TEXT NOT NULL, salary INTEGER NOT NULL,
    valid_from TEXT NOT NULL, valid_to TEXT NOT NULL, op TEXT NOT NULL
);
CREATE TRIGGER trg_emp_hist_upd AFTER UPDATE ON employees BEGIN
    INSERT INTO employees_history (emp_id, name, title, salary, valid_from, valid_to, op)
    VALUES (OLD.emp_id, OLD.name, OLD.title, OLD.salary, OLD.updated_at, NEW.updated_at, 'U');
END;
CREATE TRIGGER trg_emp_hist_del AFTER DELETE ON employees BEGIN
    INSERT INTO employees_history (emp_id, name, title, salary, valid_from, valid_to, op)
    VALUES (OLD.emp_id, OLD.name, OLD.title, OLD.salary, OLD.updated_at, (SELECT v FROM ctx WHERE k = 'now'), 'D');
END;
INSERT INTO employees VALUES (1, 'Dana', 'Analyst', 50000, '2025-01-01'), (2, 'Eli', 'Engineer', 70000, '2025-01-01');
UPDATE employees SET title = 'Senior Analyst', salary = 60000, updated_at = '2025-06-01' WHERE emp_id = 1;
UPDATE employees SET salary = 65000, updated_at = '2026-01-01' WHERE emp_id = 1;
UPDATE ctx SET v = '2026-01-10' WHERE k = 'now';
DELETE FROM employees WHERE emp_id = 2;
SELECT hist_id, emp_id, name, title, salary, valid_from, valid_to, op FROM employees_history ORDER BY hist_id;

hist_id | emp_id | name | title          | salary | valid_from | valid_to   | op
--------+--------+------+----------------+--------+------------+------------+---
1       | 1      | Dana | Analyst        | 50000  | 2025-01-01 | 2025-06-01 | U 
2       | 1      | Dana | Senior Analyst | 60000  | 2025-06-01 | 2026-01-01 | U 
3       | 2      | Eli  | Engineer       | 70000  | 2025-01-01 | 2026-01-10 | D 
(3 rows)



In [10]:
%%sql
SELECT 'history' AS source, emp_id, title, salary, valid_from, valid_to FROM employees_history WHERE emp_id = 1
UNION ALL
SELECT 'current', emp_id, title, salary, updated_at, 'open' FROM employees WHERE emp_id = 1
ORDER BY valid_from;
SELECT salary AS salary_on_2025_08_01 FROM employees_history
WHERE emp_id = 1 AND valid_from <= '2025-08-01' AND '2025-08-01' < valid_to;

source  | emp_id | title          | salary | valid_from | valid_to  
--------+--------+----------------+--------+------------+-----------
history | 1      | Analyst        | 50000  | 2025-01-01 | 2025-06-01
history | 1      | Senior Analyst | 60000  | 2025-06-01 | 2026-01-01
current | 1      | Senior Analyst | 65000  | 2026-01-01 | open      
(3 rows)

salary_on_2025_08_01
--------------------
60000               
(1 row)



## 07. Restoring from history
- Deleted by mistake? The last `D` row in history has the full old row. Insert it back (new `updated_at` = today).
- Want to undo a change? Update the live row from the latest history row. That update is itself recorded, so history keeps growing and nothing is rewritten.
- Below: bring Eli back, and roll Dana's 65000 salary back to the previous 60000. Dana then has three history rows.

In [11]:
%%sql
INSERT INTO employees (emp_id, name, title, salary, updated_at)
SELECT emp_id, name, title, salary, (SELECT v FROM ctx WHERE k = 'now')
FROM employees_history WHERE emp_id = 2 AND op = 'D' ORDER BY hist_id DESC LIMIT 1;
UPDATE employees
SET salary = (SELECT salary FROM employees_history WHERE emp_id = 1 ORDER BY hist_id DESC LIMIT 1),
    updated_at = (SELECT v FROM ctx WHERE k = 'now')
WHERE emp_id = 1;
SELECT emp_id, name, title, salary, updated_at FROM employees ORDER BY emp_id;
SELECT emp_id, count(*) AS history_rows FROM employees_history GROUP BY emp_id ORDER BY emp_id;

emp_id | name | title          | salary | updated_at
-------+------+----------------+--------+-----------
1      | Dana | Senior Analyst | 60000  | 2026-01-10
2      | Eli  | Engineer       | 70000  | 2026-01-10
(2 rows)

emp_id | history_rows
-------+-------------
1      | 3           
2      | 1           
(2 rows)



## 08. Valid-from / valid-to rows (SCD type 2)
- Instead of a separate history table, keep every version in ONE table. Each row says when it was true: `valid_from` (inclusive) to `valid_to` (exclusive).
- The current version has the sentinel `valid_to = '9999-12-31'`. A real far-future date compares correctly as text, unlike NULL, which would need special cases.
- Change procedure as two plain SQL steps. Step 1: close the current row if the value really changes. Step 2: insert the new row only if no current row remains.
- Because step 2 checks for an open row, repeating the same change does nothing. The procedure is idempotent.
- Customer 1 moves Mumbai, then Pune on 2025-03-01, then Delhi on 2025-09-15.

In [12]:
%%sql
CREATE TABLE customer_addr (
    sk          INTEGER PRIMARY KEY,
    customer_id INTEGER NOT NULL,
    city        TEXT NOT NULL,
    valid_from  TEXT NOT NULL,
    valid_to    TEXT NOT NULL DEFAULT '9999-12-31',
    CHECK (valid_from < valid_to)
);
INSERT INTO customer_addr (customer_id, city, valid_from) VALUES (1, 'Mumbai', '2024-01-01'), (2, 'Chennai', '2024-06-01');
-- change 1: customer 1 moves to Pune on 2025-03-01
UPDATE customer_addr SET valid_to = '2025-03-01' WHERE customer_id = 1 AND valid_to = '9999-12-31' AND city <> 'Pune';
INSERT INTO customer_addr (customer_id, city, valid_from)
SELECT 1, 'Pune', '2025-03-01' WHERE NOT EXISTS (SELECT 1 FROM customer_addr WHERE customer_id = 1 AND valid_to = '9999-12-31');
-- change 2: customer 1 moves to Delhi on 2025-09-15
UPDATE customer_addr SET valid_to = '2025-09-15' WHERE customer_id = 1 AND valid_to = '9999-12-31' AND city <> 'Delhi';
INSERT INTO customer_addr (customer_id, city, valid_from)
SELECT 1, 'Delhi', '2025-09-15' WHERE NOT EXISTS (SELECT 1 FROM customer_addr WHERE customer_id = 1 AND valid_to = '9999-12-31');
SELECT sk, customer_id, city, valid_from, valid_to FROM customer_addr ORDER BY customer_id, valid_from;

sk | customer_id | city    | valid_from | valid_to  
---+-------------+---------+------------+-----------
1  | 1           | Mumbai  | 2024-01-01 | 2025-03-01
3  | 1           | Pune    | 2025-03-01 | 2025-09-15
4  | 1           | Delhi   | 2025-09-15 | 9999-12-31
2  | 2           | Chennai | 2024-06-01 | 9999-12-31
(4 rows)



In [13]:
%%sql
-- repeat change 2 (same city, same date): both steps must do nothing
UPDATE customer_addr SET valid_to = '2025-09-15' WHERE customer_id = 1 AND valid_to = '9999-12-31' AND city <> 'Delhi';
SELECT changes() AS closed_rows;
INSERT INTO customer_addr (customer_id, city, valid_from)
SELECT 1, 'Delhi', '2025-09-15' WHERE NOT EXISTS (SELECT 1 FROM customer_addr WHERE customer_id = 1 AND valid_to = '9999-12-31');
SELECT changes() AS inserted_rows;
SELECT count(*) AS total_rows FROM customer_addr;

closed_rows
-----------
0          
(1 row)

inserted_rows
-------------
0            
(1 row)

total_rows
----------
4         
(1 row)



## 09. Point-in-time (as-of) queries
- "Where did customer 1 live on date d?" = the row where `valid_from <= d AND d < valid_to`. Exactly one row, because the ranges do not overlap.
- The half-open range matters on the boundary: on `2025-03-01` the answer is Pune, not Mumbai. With `BETWEEN` both rows would match.
- As-of JOIN: attach to each fact the dimension row that was valid on the fact's own date. This is the classic reason for SCD type 2: old orders keep the old city.

In [14]:
%%sql
WITH dates(d) AS (VALUES ('2024-02-01'), ('2025-02-28'), ('2025-03-01'), ('2025-12-31'))
SELECT d.d AS as_of, a.customer_id, a.city
FROM dates d JOIN customer_addr a ON a.valid_from <= d.d AND d.d < a.valid_to
WHERE a.customer_id = 1 ORDER BY d.d;
CREATE TABLE cust_orders (order_id INTEGER PRIMARY KEY, customer_id INTEGER NOT NULL, order_date TEXT NOT NULL);
INSERT INTO cust_orders VALUES (1, 1, '2024-02-10'), (2, 1, '2025-04-01'), (3, 1, '2025-10-01'), (4, 2, '2025-01-01');
SELECT o.order_id, o.order_date, a.city AS city_at_order_time
FROM cust_orders o JOIN customer_addr a
  ON a.customer_id = o.customer_id AND a.valid_from <= o.order_date AND o.order_date < a.valid_to
ORDER BY o.order_id;

as_of      | customer_id | city  
-----------+-------------+-------
2024-02-01 | 1           | Mumbai
2025-02-28 | 1           | Mumbai
2025-03-01 | 1           | Pune  
2025-12-31 | 1           | Delhi 
(4 rows)

order_id | order_date | city_at_order_time
---------+------------+-------------------
1        | 2024-02-10 | Mumbai            
2        | 2025-04-01 | Pune              
3        | 2025-10-01 | Delhi             
4        | 2025-01-01 | Chennai           
(4 rows)



## 10. Guarding against overlaps (and spotting gaps)
- Nothing in plain SQL stops two rows for the same customer from covering the same day. Then "as of" returns two rows and joins duplicate facts.
- Two ranges overlap when `a.from < b.to AND b.from < a.to`. A `BEFORE INSERT/UPDATE` trigger checks that against existing rows and aborts.
- Ranges that merely touch (one ends the day the next begins) do not overlap, so the close-then-insert procedure still works.
- Gaps (a period with no row) are legal but often a bug. `lead()` finds them: next row's start should equal this row's end. See 03-Window-Functions/01-Window-Functions.

In [15]:
%%sql
CREATE TRIGGER trg_addr_no_overlap_ins BEFORE INSERT ON customer_addr
WHEN EXISTS (SELECT 1 FROM customer_addr WHERE customer_id = NEW.customer_id AND valid_from < NEW.valid_to AND NEW.valid_from < valid_to)
BEGIN SELECT RAISE(ABORT, 'overlapping validity range for this customer'); END;
CREATE TRIGGER trg_addr_no_overlap_upd BEFORE UPDATE OF customer_id, valid_from, valid_to ON customer_addr
WHEN EXISTS (SELECT 1 FROM customer_addr WHERE customer_id = NEW.customer_id AND sk <> OLD.sk AND valid_from < NEW.valid_to AND NEW.valid_from < valid_to)
BEGIN SELECT RAISE(ABORT, 'overlapping validity range for this customer'); END;

In [16]:
%%sql expect_error
INSERT INTO customer_addr (customer_id, city, valid_from, valid_to) VALUES (1, 'Goa', '2025-02-01', '2025-04-01');
UPDATE customer_addr SET valid_to = '2025-12-01' WHERE customer_id = 1 AND city = 'Mumbai';

ERROR: overlapping validity range for this customer

ERROR: overlapping validity range for this customer



In [17]:
%%sql
INSERT INTO customer_addr (customer_id, city, valid_from, valid_to) VALUES (3, 'Kochi', '2024-01-01', '2024-06-01');
INSERT INTO customer_addr (customer_id, city, valid_from) VALUES (3, 'Jaipur', '2024-07-01');
SELECT customer_id, valid_to AS gap_starts, next_from AS gap_ends
FROM (SELECT customer_id, valid_to, lead(valid_from) OVER (PARTITION BY customer_id ORDER BY valid_from) AS next_from FROM customer_addr)
WHERE next_from IS NOT NULL AND next_from <> valid_to
ORDER BY customer_id;

customer_id | gap_starts | gap_ends  
------------+------------+-----------
3           | 2024-06-01 | 2024-07-01
(1 row)



## 11. A current-row view
- Most queries only want today's version. Hide the sentinel behind a view: `v_customer_current`.
- A partial unique index `WHERE valid_to = '9999-12-31'` makes "at most one current row per customer" a hard rule, a second safety net next to the overlap trigger.
- Keep the view name honest: if the sentinel ever changes, only the view and index change.

In [18]:
%%sql
CREATE VIEW v_customer_current AS SELECT customer_id, city, valid_from AS since FROM customer_addr WHERE valid_to = '9999-12-31';
CREATE UNIQUE INDEX ux_addr_one_current ON customer_addr(customer_id) WHERE valid_to = '9999-12-31';
SELECT customer_id, city, since FROM v_customer_current ORDER BY customer_id;

customer_id | city    | since     
------------+---------+-----------
1           | Delhi   | 2025-09-15
2           | Chennai | 2024-06-01
3           | Jaipur  | 2024-07-01
(3 rows)



## 12. Optimistic locking with a version column
- Problem: two people open the same record, both edit, the second save silently overwrites the first (lost update).
- Fix without locks: a `version` number. Read it with the row. Save with `WHERE id = ? AND version = ?` and `SET version = version + 1`.
- If `changes()` is 1 you won. If it is 0 someone saved first: re-read, merge, retry.
- "Optimistic" because it assumes conflicts are rare and only checks at save time. It needs no long-held database locks. See 06-Data-Modification/02-Transactions for the pessimistic alternatives.

In [19]:
%%sql
CREATE TABLE docs (doc_id INTEGER PRIMARY KEY, title TEXT NOT NULL, body TEXT NOT NULL, version INTEGER NOT NULL DEFAULT 1);
INSERT INTO docs (doc_id, title, body) VALUES (1, 'Plan', 'draft');
SELECT doc_id, body, version FROM docs;
UPDATE docs SET body = 'edit by Asha', version = version + 1 WHERE doc_id = 1 AND version = 1;
SELECT changes() AS asha_rows_changed;
UPDATE docs SET body = 'edit by Ben', version = version + 1 WHERE doc_id = 1 AND version = 1;
SELECT changes() AS ben_rows_changed;
SELECT doc_id, body, version FROM docs;
UPDATE docs SET body = 'edit by Ben (after re-reading)', version = version + 1 WHERE doc_id = 1 AND version = 2;
SELECT changes() AS ben_retry_rows_changed;
SELECT doc_id, body, version FROM docs;

doc_id | body  | version
-------+-------+--------
1      | draft | 1      
(1 row)

asha_rows_changed
-----------------
1                
(1 row)

ben_rows_changed
----------------
0               
(1 row)

doc_id | body         | version
-------+--------------+--------
1      | edit by Asha | 2      
(1 row)

ben_retry_rows_changed
----------------------
1                     
(1 row)

doc_id | body                           | version
-------+--------------------------------+--------
1      | edit by Ben (after re-reading) | 3      
(1 row)



## 13. Event-sourcing-lite: an append-only ledger
- Do not store "the balance" and update it. Store every movement as an immutable row; the balance is the sum.
- Triggers refuse `UPDATE` and `DELETE`, so history cannot be rewritten by accident.
- A mistake is fixed by adding a reversing entry (negative amount, pointing at the entry it reverses). Both stay visible forever.
- Running balance with a window function: `sum(amount_cents) OVER (PARTITION BY account ORDER BY entry_id ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW)`.
- Account ACC1 has a mistaken 900 purchase (entry 3), reversed by entry 4.

In [20]:
%%sql
CREATE TABLE ledger (
    entry_id     INTEGER PRIMARY KEY AUTOINCREMENT,
    account      TEXT NOT NULL,
    entry_date   TEXT NOT NULL,
    amount_cents INTEGER NOT NULL,
    memo         TEXT NOT NULL,
    reverses     INTEGER REFERENCES ledger(entry_id)
);
CREATE TRIGGER trg_ledger_no_update BEFORE UPDATE ON ledger BEGIN SELECT RAISE(ABORT, 'ledger is append-only: add a reversing entry'); END;
CREATE TRIGGER trg_ledger_no_delete BEFORE DELETE ON ledger BEGIN SELECT RAISE(ABORT, 'ledger is append-only: add a reversing entry'); END;
INSERT INTO ledger (account, entry_date, amount_cents, memo, reverses) VALUES
    ('ACC1', '2026-01-02', 10000, 'deposit', NULL),
    ('ACC1', '2026-01-03', -2500, 'rent share', NULL),
    ('ACC1', '2026-01-04', -900, 'coffee machine (wrong account)', NULL),
    ('ACC1', '2026-01-05', 900, 'reversal of entry 3', 3),
    ('ACC1', '2026-01-06', 5000, 'deposit', NULL),
    ('ACC2', '2026-01-02', 3000, 'deposit', NULL),
    ('ACC2', '2026-01-04', -1000, 'withdrawal', NULL);
SELECT entry_id, account, entry_date, amount_cents, memo,
       sum(amount_cents) OVER (PARTITION BY account ORDER BY entry_id ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW) AS balance
FROM ledger ORDER BY account, entry_id;

entry_id | account | entry_date | amount_cents | memo                           | balance
---------+---------+------------+--------------+--------------------------------+--------
1        | ACC1    | 2026-01-02 | 10000        | deposit                        | 10000  
2        | ACC1    | 2026-01-03 | -2500        | rent share                     | 7500   
3        | ACC1    | 2026-01-04 | -900         | coffee machine (wrong account) | 6600   
4        | ACC1    | 2026-01-05 | 900          | reversal of entry 3            | 7500   
5        | ACC1    | 2026-01-06 | 5000         | deposit                        | 12500  
6        | ACC2    | 2026-01-02 | 3000         | deposit                        | 3000   
7        | ACC2    | 2026-01-04 | -1000        | withdrawal                     | 2000   
(7 rows)



In [21]:
%%sql expect_error
UPDATE ledger SET amount_cents = 0 WHERE entry_id = 3;
DELETE FROM ledger WHERE entry_id = 3;

ERROR: ledger is append-only: add a reversing entry

ERROR: ledger is append-only: add a reversing entry



## 14. Ledger queries: balances, as-of balances, reversals
- Current balance per account = `GROUP BY account` with `sum`.
- Balance as of a date = the same sum with `entry_date <= d`. No snapshots needed for small ledgers; big ones add periodic snapshot rows and sum only what came after.
- Which entries were cancelled? A self-join on `reverses` lists them: entry 3 was reversed by entry 4. An anti-join (`NOT EXISTS`) lists the ordinary entries that were never cancelled (5 of them; the reversal row itself is not counted).
- Net of a reversed entry and its reversal is 0, so balances stay correct without editing anything.

In [22]:
%%sql
SELECT account, sum(amount_cents) AS balance FROM ledger GROUP BY account ORDER BY account;
SELECT account, sum(amount_cents) AS balance_on_2026_01_04 FROM ledger WHERE entry_date <= '2026-01-04' GROUP BY account ORDER BY account;
SELECT e.entry_id, e.memo, r.entry_id AS reversed_by
FROM ledger e JOIN ledger r ON r.reverses = e.entry_id ORDER BY e.entry_id;
SELECT count(*) AS entries_not_cancelled
FROM ledger e WHERE e.reverses IS NULL AND NOT EXISTS (SELECT 1 FROM ledger r WHERE r.reverses = e.entry_id);

account | balance
--------+--------
ACC1    | 12500  
ACC2    | 2000   
(2 rows)

account | balance_on_2026_01_04
--------+----------------------
ACC1    | 6600                 
ACC2    | 2000                 
(2 rows)

entry_id | memo                           | reversed_by
---------+--------------------------------+------------
3        | coffee machine (wrong account) | 4          
(1 row)

entries_not_cancelled
---------------------
5                    
(1 row)



## 15. Erase vs anonymise (GDPR-style requests)
- A person asks you to delete their data. Two options. Hard erase: remove the rows. Anonymise: keep the row but overwrite everything that identifies the person.
- Hard erase is simplest when nothing else depends on the person (member 3 below). Child rows with `ON DELETE CASCADE` go with it.
- If other rows must stay (orders for the accounts), the foreign key blocks the delete. Anonymise instead: placeholder email (still unique), no name, no phone, and an `erased_at` stamp. Totals and counts stay right.
- Do not forget the copies: event payloads (scrub with `json_remove`), audit logs, history tables, exports, search indexes, and backups. Free pages in the file can keep old bytes until `VACUUM` (or `PRAGMA secure_delete`).
- Pick the approach with legal advice. SQL cannot answer what you must keep for the law.

In [23]:
%%sql
CREATE TABLE members (member_id INTEGER PRIMARY KEY, email TEXT NOT NULL UNIQUE, name TEXT NOT NULL, phone TEXT, erased_at TEXT);
CREATE TABLE member_orders (order_id INTEGER PRIMARY KEY, member_id INTEGER NOT NULL REFERENCES members(member_id), total_cents INTEGER NOT NULL);
CREATE TABLE member_events (event_id INTEGER PRIMARY KEY, member_id INTEGER NOT NULL REFERENCES members(member_id) ON DELETE CASCADE, kind TEXT NOT NULL, payload TEXT NOT NULL);
INSERT INTO members (member_id, email, name, phone) VALUES (1, 'asha@example.com', 'Asha Rao', '555-0101'), (2, 'ben@example.com', 'Ben Carter', '555-0102'), (3, 'chen@example.com', 'Chen Wei', NULL);
INSERT INTO member_orders VALUES (1, 1, 2500), (2, 1, 900), (3, 2, 1200);
INSERT INTO member_events VALUES
    (1, 1, 'signup', '{"email":"asha@example.com","name":"Asha Rao","plan":"free"}'),
    (2, 1, 'login',  '{"email":"asha@example.com","device":"phone"}'),
    (3, 3, 'signup', '{"email":"chen@example.com","name":"Chen Wei","plan":"free"}');

In [24]:
%%sql expect_error
DELETE FROM members WHERE member_id = 1;

ERROR: FOREIGN KEY constraint failed



In [25]:
%%sql
DELETE FROM members WHERE member_id = 3;
SELECT (SELECT count(*) FROM members) AS members_left, (SELECT count(*) FROM member_events WHERE member_id = 3) AS chen_events_left;
UPDATE members SET email = 'erased-' || member_id || '@invalid.example', name = 'Erased member', phone = NULL,
                   erased_at = (SELECT v FROM ctx WHERE k = 'now')
WHERE member_id = 1;
UPDATE member_events SET payload = json_remove(payload, '$.email', '$.name') WHERE member_id = 1;
SELECT member_id, email, name, phone, erased_at FROM members ORDER BY member_id;
SELECT event_id, kind, payload FROM member_events ORDER BY event_id;
SELECT member_id, sum(total_cents) AS total_cents FROM member_orders GROUP BY member_id ORDER BY member_id;

members_left | chen_events_left
-------------+-----------------
2            | 0               
(1 row)

member_id | email                    | name          | phone    | erased_at 
----------+--------------------------+---------------+----------+-----------
1         | erased-1@invalid.example | Erased member | NULL     | 2026-01-10
2         | ben@example.com          | Ben Carter    | 555-0102 | NULL      
(2 rows)

event_id | kind   | payload           
---------+--------+-------------------
1        | signup | {"plan":"free"}   
2        | login  | {"device":"phone"}
(2 rows)

member_id | total_cents
----------+------------
1         | 3400       
2         | 1200       
(2 rows)



## 16. Bitemporal data, briefly
- Two different times: VALID time (when it was true in the real world) and TRANSACTION time (when the database knew about it).
- Needed when you must answer both "what is the price on 15 Feb?" and "what did we think the price on 15 Feb was, back on 20 Feb, before we found the mistake?".
- Each row gets four dates: `valid_from/valid_to` and `known_from/known_to`. A correction never overwrites old facts: it only closes the old row's `known_to` and adds new rows.
- Story: price 100 from 2025-01-01. On 2025-03-01 we learn that it was really 90 from 2025-02-01.
- It is heavy. Use it only when you truly need both questions (finance, insurance, regulation).

In [26]:
%%sql
CREATE TABLE price_bt (
    row_id INTEGER PRIMARY KEY, sku TEXT NOT NULL, price INTEGER NOT NULL,
    valid_from TEXT NOT NULL, valid_to TEXT NOT NULL, known_from TEXT NOT NULL, known_to TEXT NOT NULL
);
INSERT INTO price_bt VALUES
    (1, 'W', 100, '2025-01-01', '9999-12-31', '2025-01-01', '2025-03-01'),
    (2, 'W', 100, '2025-01-01', '2025-02-01', '2025-03-01', '9999-12-31'),
    (3, 'W',  90, '2025-02-01', '9999-12-31', '2025-03-01', '9999-12-31');
SELECT 'believed on 2025-02-20' AS question, price FROM price_bt
WHERE valid_from <= '2025-02-15' AND '2025-02-15' < valid_to AND known_from <= '2025-02-20' AND '2025-02-20' < known_to
UNION ALL
SELECT 'known today', price FROM price_bt
WHERE valid_from <= '2025-02-15' AND '2025-02-15' < valid_to AND known_from <= '2026-01-05' AND '2026-01-05' < known_to;

question               | price
-----------------------+------
believed on 2025-02-20 | 100  
known today            | 90   
(2 rows)



## 17. Choosing a pattern
- Need "undo delete" and a hidden-but-kept row: soft delete (`deleted_at` + view + partial unique index).
- Need "who changed what, when" for many tables with little effort: audit-log triggers writing JSON.
- Need typed queries over old versions of one table: history table (or SCD type 2).
- Need "state as of any date" joined to facts: SCD type 2 with half-open ranges, an overlap guard and a current view.
- Need to stop lost updates between editors: version column with optimistic locking.
- Need trustworthy money or stock movements: append-only ledger, reversing entries, running balance.
- Need to forget a person: anonymise or erase, and chase every copy (events, audit, history, backups).
- Each pattern costs storage and complexity. Add only the ones you can name a question for.